# Stage 4 — Attention Rollout Visualization

Visualising which image patches the trained ViT attends to when making a classification decision.

**Attention rollout** (Abnar & Zuidema, 2020) estimates total attention flow from input to output
by multiplying attention matrices across layers:

```
rollout = A_L @ A_{L-1} @ ... @ A_1
```

where each `A_i = (attn_i + I) / 2` accounts for residual connections passing information
through unchanged. Row 0 of the result (the CLS token) shows how much each input patch
contributed to the final classification.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

## 1. Architecture (modified to capture attention weights)

`scaled_dot_product_attention` now returns `(output, weights)` so that
`MultiHeadAttention` can store them as `self.attn_weights` after each forward pass.

In [ ]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores  = Q @ K.transpose(-2, -1) / (d_k ** 0.5)  # [*, N, N]
    weights = torch.softmax(scores, dim=-1)             # [*, N, N]
    return weights @ V, weights                         # output + weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X):
        B, N, D = X.shape

        Q = self.W_Q(X)  # [B, N, D]
        K = self.W_K(X)  # [B, N, D]
        V = self.W_V(X)  # [B, N, D]

        Q = Q.reshape(B, N, self.num_heads, self.d_k).transpose(1, 2)  # [B, h, N, d_k]
        K = K.reshape(B, N, self.num_heads, self.d_k).transpose(1, 2)
        V = V.reshape(B, N, self.num_heads, self.d_k).transpose(1, 2)

        out, attn_weights = scaled_dot_product_attention(Q, K, V)
        self.attn_weights = attn_weights  # [B, num_heads, N, N] — stored for rollout

        out = out.transpose(1, 2).reshape(B, N, D)
        return self.W_O(out)  # [B, N, D]


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn  = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )

    def forward(self, X):
        X = self.norm1(X + self.attn(X))
        X = self.norm2(X + self.ffn(X))
        return X


class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, d_model):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, X):
        X = self.proj(X)                                         # [B, d_model, h, w]
        X = X.reshape(X.shape[0], X.shape[1], self.num_patches)  # [B, d_model, num_patches]
        return X.transpose(-2, -1)                               # [B, num_patches, d_model]


class ViT(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, d_model, num_heads, num_layers, num_classes):
        super().__init__()
        num_patches = (img_size // patch_size) ** 2

        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, d_model)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_embed   = nn.Parameter(torch.randn(1, num_patches + 1, d_model))

        self.blocks     = nn.ModuleList([
            TransformerBlock(d_model, num_heads) for _ in range(num_layers)
        ])
        self.norm        = nn.LayerNorm(d_model)
        self.classifier  = nn.Linear(d_model, num_classes)

    def forward(self, X):
        B   = X.shape[0]
        X   = self.patch_embed(X)               # [B, num_patches, d_model]
        cls = self.cls_token.expand(B, -1, -1)  # [B, 1, d_model]
        X   = torch.cat([cls, X], dim=1)         # [B, num_patches+1, d_model]
        X   = X + self.pos_embed

        for block in self.blocks:
            X = block(X)

        cls_out = self.norm(X)[:, 0]             # [B, d_model]
        return self.classifier(cls_out)          # [B, num_classes]

## 2. Load Trained Model

Hyperparameters must match the checkpoint exactly — the positional embedding shape
`[1, num_patches+1, d_model]` is baked into the weights.

In [ ]:
IMG_SIZE    = 32
PATCH_SIZE  = 4
D_MODEL     = 256
NUM_HEADS   = 8
NUM_LAYERS  = 6
NUM_CLASSES = 10

CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = ViT(IMG_SIZE, PATCH_SIZE, 3, D_MODEL, NUM_HEADS, NUM_LAYERS, NUM_CLASSES)
model.load_state_dict(torch.load('models/vit_cifar10_p4_e50.pth', weights_only=True))
model.to(device).eval()
print('Model loaded.')

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# quick sanity check on one image
img, label = test_set[0]
inp  = img.unsqueeze(0).to(device)  # [1, 3, 32, 32]
with torch.no_grad():
    pred = model(inp).argmax(dim=-1).item()

print(f"True: {CLASSES[label]}  |  Predicted: {CLASSES[pred]}")

## 3. Attention Rollout

After a forward pass, each `TransformerBlock` stores its attention weights at
`block.attn.attn_weights` — shape `[B, num_heads, N, N]`.

Rollout:
1. Average over heads → one `[N, N]` matrix per layer
2. Add identity to model residual skip: `A = (A + I) / 2`
3. Multiply matrices layer-by-layer: total flow = `A_L @ ... @ A_1`
4. Row 0 of the result = CLS token's attention over all input patches

In [ ]:
def attention_rollout(attn_maps):
    """attn_maps: list of [1, num_heads, N, N] tensors, one per layer."""
    # average over heads: [num_layers, N, N]
    attn = torch.stack([a.squeeze(0).mean(0) for a in attn_maps])

    # residual connections pass information unchanged, so we blend with identity
    I    = torch.eye(attn.shape[-1], device=attn.device)
    attn = (attn + I) / 2

    # chain attention through layers
    result = attn[0]
    for i in range(1, len(attn)):
        result = attn[i] @ result

    return result  # [N, N]

## 4. Visualize

Extract CLS attention (row 0, skipping the CLS-to-CLS entry at column 0),
reshape to the patch grid, upsample to image resolution, and overlay.

In [ ]:
def visualize_attention(img_tensor, label, pred, attn_maps, img_size, patch_size):
    num_patches_side = img_size // patch_size

    rollout  = attention_rollout(attn_maps)
    cls_attn = rollout[0, 1:].cpu().numpy()                      # [num_patches]
    heatmap  = cls_attn.reshape(num_patches_side, num_patches_side)

    # upsample patch grid to image resolution
    heatmap_up = F.interpolate(
        torch.tensor(heatmap).unsqueeze(0).unsqueeze(0),
        size=(img_size, img_size),
        mode='bilinear',
    ).squeeze().numpy()

    # undo normalisation for display
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = np.clip((img_np * 0.5) + 0.5, 0, 1)

    correct = '(correct)' if label == pred else '(wrong)'
    title   = f"{CLASSES[label]} {correct}\npred: {CLASSES[pred]}"

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img_np)
    axes[0].set_title(f'Input\n{title}')
    axes[1].imshow(heatmap, cmap='viridis')
    axes[1].set_title(f'Attention ({num_patches_side}x{num_patches_side} grid)')
    axes[2].imshow(img_np)
    axes[2].imshow(heatmap_up, cmap='viridis', alpha=0.5)
    axes[2].set_title('Overlay')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
img, label = test_set[0]
inp = img.unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(inp)

pred      = logits.argmax(dim=-1).item()
attn_maps = [block.attn.attn_weights for block in model.blocks]

visualize_attention(img, label, pred, attn_maps, IMG_SIZE, PATCH_SIZE)

In [ ]:
for idx in [1, 5, 10, 50]:
    img, label = test_set[idx]
    inp = img.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(inp)

    pred      = logits.argmax(dim=-1).item()
    attn_maps = [block.attn.attn_weights for block in model.blocks]

    visualize_attention(img, label, pred, attn_maps, IMG_SIZE, PATCH_SIZE)